# momentum-signal-scanner (project by Bervyn Wong)
## RSI and MACD long-short rules:
- scanned across a universe of 8 large-cap US stocks
- ranked according to risk-adjusted return

**My background** · BSc(Hons.) Mathematics, NUS & MSc Quantitative Finance, SMU

> I first worked on momentum and technical-signal research during a quantitative trading internship in 2022. This notebook is my own rebuild of that method on public market data from yahoo finance. To clarify, there is no employer code, data or strategy in it. I rebuilt it to relearn the ideas properly and to be able to make it my own self-directed project.

> **The question.** If I run two classic indicator rules over eight large stocks from 2015 to 2021, which ticker-rule pairs might be profitable, and how should I rank them?

> **Approach.** Rank by Sharpe ratio and expected risk-adjusted return, not by win rate. The rule with the best win rate in this scan (TSM on RSI, 88% of trades won) lost more than half its capital.

The structure follows the six stages of a rule (price → indicator → signal → detected → executed → PnL), and the indicators built are handrolled by yours truly.

## <a id = "top">Learning pointers</a>

1. [Setup](#p1) · *every setting in one cell*
2. [The data](#p2) · *know what one row is before computing anything*
3. [Two indicators, built by hand](#p3) · *RSI is a share of recent gains; MACD is a gap between two averages*
4. [From indicator to trade: the six stages](#p4) · *one `shift(1)` separates a backtest from a fantasy*
5. [Scoring a book](#p5) · *win rate counts wins, expectancy weighs them*
6. [The scan](#p6) · *rank by Sharpe, not by hit rate*
7. [Inside two books](#p7) · *what the winner and the high-win-rate loser look like*
8. [What I learned](#p8)
9. [What I changed from the 2022 version, and the limits](#p10)

---
## <a id = "p1">1. Setup</a> [back to top](#top)

Everything I might want to change lives in one cell: the universe, the dates, the cost and the indicator settings. The functions below take these as parameters, so nothing is buried inside the logic.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import lets_plot
from lets_plot import *

# LetsPlot.setup_html(no_js = True) # no_js renders every chart as static SVG, so the plots also show on GitHub.
LetsPlot.setup_html(no_js=True)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 100)

print("python   :", sys.version.split()[0])
print("numpy    :", np.__version__)
print("pandas   :", pd.__version__)
print("lets-plot:", lets_plot.__version__)

python   : 3.13.13
numpy    : 2.5.3
pandas   : 3.0.5
lets-plot: 4.11.0


In [2]:
# universe and period
UNIVERSE = ("AAPL", "MSFT", "AMZN", "GOOGL", "NVDA", "JPM", "TSLA", "TSM")
START, END = "2015-01-01", "2021-12-31"

# fixed parameters
TRADING_DAYS = 252
RISK_FREE = 0.02     # annual rate, subtracted inside the Sharpe ratio
COST_BP = 5.0        # basis points paid per unit change in position

# textbook indicator settings, using default parameters.
RSI_WINDOW, RSI_OVERSOLD, RSI_OVERBOUGHT = 14, 30, 70
MACD_FAST, MACD_SLOW, MACD_SIGNAL = 12, 26, 9

REFRESH = False
DATA_DIR = Path("data")
CACHE_FILE = DATA_DIR / f"universe_{'-'.join(sorted(UNIVERSE))}_{START}_{END}.csv"
FIGURE_DIR = Path("docs") / "img"

In [3]:
# One palette and one theme for every chart, in the QF627 style.
NAVY, TEAL, RED, GREEN, GREY = "#1B3556", "#2B7A9B", "#C41230", "#2E8B57", "#8C8C8C"

# one assigned colour per ticker for data visualisation purposes
TICKER_COLORS = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9", "#D55E00", "#000000", "#999999"]

house_theme = (
            theme_minimal() 
            + theme(
            legend_position = "top",
            legend_title = element_blank(),
            plot_title = element_text(face = "bold"),
            )
)

---
## <a id = "p2">2. The data</a> [back to top](#top)

### Load

The repo ships a cached CSV so the notebook runs offline and gives the same numbers every time. `yfinance` is only needed if I set `REFRESH = True`, which is why it is imported inside the download function and not at the top.

In [4]:
def download_closes(tickers, start, end) -> pd.DataFrame:
    import yfinance as yf
    raw = yf.download(list(tickers), start=start, end=end, auto_adjust=True, progress=False)

    return raw["Close"].dropna(how="all")

def load_closes(cache_file, tickers, start, end, refresh=False) -> pd.DataFrame:
    # read the cache; download and overwrite it only when asked.
    if refresh:
        cache_file.parent.mkdir(exist_ok=True)
        download_closes(tickers, start, end).to_csv(cache_file)

    return pd.read_csv(cache_file, index_col="Date", parse_dates=True)


closes = load_closes(CACHE_FILE, UNIVERSE, START, END, REFRESH)
closes.tail().round(2)

,AAPL,MSFT,AMZN,GOOGL,NVDA,JPM,TSLA,TSM
Date,,,,,,,,
2021-12-03,161.84,323.01,3389.79,2840.03,306.93,158.29,1014.97,119.33
2021-12-06,165.32,326.19,3427.37,2863.10,300.37,160.16,1009.01,119.14
2021-12-07,171.18,334.92,3523.29,2945.39,324.27,162.57,1051.75,122.32
2021-12-08,175.08,334.97,3523.16,2963.73,318.26,160.71,1068.96,121.80
2021-12-09,174.56,333.10,3483.42,2952.77,304.90,160.46,1003.80,119.50


**What one row is.** One NYSE trading day. Each column is that ticker's closing price in US dollars, **adjusted** for splits and dividends (`auto_adjust = True`).

The adjustment matters more than it looks. QF627 Week 4 showed that on an ex-dividend day the quoted close drops by the dividend, and the RSI books that drop as a loss even though no holder lost anything. Splits are worse: AAPL's 4-for-1 split in August 2020 would look like a 75% crash on the raw close. So every indicator here runs on adjusted closes.

### Audit

Before any indicator touches the panel, I check the things that would silently corrupt it: the grain (one row per date), ordering, gaps, missing values and prices that cannot be true.

In [5]:
# If any of these is non-zero, every number below is suspect.
audit = pd.Series({
    "trading days": len(closes),
    "tickers": closes.shape[1],
    "duplicate dates": int(closes.index.duplicated().sum()),
    "dates out of order": int(not closes.index.is_monotonic_increasing),
    "weekend rows": int((closes.index.dayofweek >= 5).sum()),
    "missing closes": int(closes.isna().sum().sum()),
    "non-positive closes": int((closes <= 0).sum().sum()),
})

audit

trading days           1748
tickers                   8
duplicate dates           0
dates out of order        0
weekend rows              0
missing closes            0
non-positive closes       0
dtype: int64

In [6]:
# Calendar days between consecutive rows.
# 1 normal day, 2 = mid-week holiday, 3 = weekend, 4 = long weekend. 
# Anything longer means missing data.
dates = closes.index.to_series()
days_between_rows = dates.diff().dt.days

gap_days = days_between_rows.value_counts().sort_index()
gap_days.name = "rows"

gap_days

Date
1.0    1370
2.0      15
3.0     315
4.0      47
Name: rows, dtype: int64

In [7]:
# A split the data provider failed to adjust shows up as a one-day move of
# -50% or -75%. The largest moves should be real news, not data errors.
daily_returns = closes.pct_change().dropna()
biggest_day = daily_returns.abs().idxmax()

largest_moves = pd.DataFrame({
    "date": biggest_day.dt.date,
    "move": [daily_returns.loc[day, ticker] for ticker, day in biggest_day.items()],
})

largest_moves.sort_values("move").round(3)

,date,move
TSLA,2020-09-08,-0.211
MSFT,2020-03-16,-0.147
TSM,2020-03-16,-0.140
AAPL,2020-03-16,-0.129
AMZN,2015-04-24,0.141
GOOGL,2015-07-17,0.163
JPM,2020-03-13,0.180
NVDA,2016-11-11,0.298


**What the audit says**

- The panel is clean: one row per trading day, in order, no weekends, no missing or impossible prices. The longest gap is 4 calendar days, which is a long weekend.
- The largest one-day moves are real events: COVID days in March 2020 (AAPL, MSFT, TSM, JPM), earnings-day jumps (AMZN in April 2015, GOOGL in July 2015, NVDA in November 2016) and TSLA's 21% drop in September 2020. None is the −50% or −75% of an unadjusted split.

| Issue | Where | Magnitude | Action | Reasoning |
|---|---|---|---|---|
| Missing, duplicate or non-positive prices | whole panel | 0 rows | none | nothing to fix |
| Unadjusted splits | AAPL, TSLA, NVDA split during the sample | 0 suspicious moves | none | `auto_adjust` already handled them |
| **Survivorship bias** | universe choice | all 8 names | **kept, flagged** | I picked eight stocks that are famous *because* they did well. A real study needs a point-in-time universe that includes the losers and delistings. This is a limit of the setup, not something I can clean away. |

### What the market did

One look at the backdrop before testing anything. If every stock went up a lot, a rule that spends half its time short starts with a handicap.

In [8]:
# Growth of $1 bought on the first day: compound the daily returns. 
returns = closes.pct_change().fillna(0.0)
growth = (1 + returns).cumprod()
growth.index.name = "date"
growth_LONG = (
    growth
    .reset_index()
    .melt(id_vars="date", var_name="ticker", value_name="growth")
    )

volatility = daily_returns.std() * np.sqrt(TRADING_DAYS)
annual_volatility = pd.DataFrame(
    {
        "ticker": volatility.index, 
        "volatility": volatility.values
    }
)

growth_plot = (
    ggplot(growth_LONG, aes(x = "date", y = "growth", color = "ticker"))
    + geom_line(size = 0.7)
    + scale_y_log10()
    + scale_color_manual(values=TICKER_COLORS)
    + labs(title = "$1 equity curve", subtitle = "log scale", x = "", y = "value of $1")
    + house_theme
)

volatility_plot = (
    ggplot(annual_volatility, aes(x ="ticker", y="volatility"))
    + geom_bar(stat="identity", fill=NAVY, width=0.7)
    + labs(title = "Annualised volatility of daily returns", x = "", y = "volatility")
    + house_theme
)

gggrid([growth_plot, volatility_plot], ncol=2) + ggsize(1200, 400)

**Reading the figure.** 2015 to 2021 was one long bull market for these names. Buy-and-hold made money on all eight, from about 3x (JPM) to about 60x (NVDA).

---
## <a id = "p3">3. Two indicators, built by hand</a> [back to top](#top)

### 3.1 RSI

**Intuition.** Picture a tug-of-war scoreboard for the last two weeks. On up days the buyers gain ground and on down days the sellers do. The RSI is the buyers' share of all the ground gained, on a 0 to 100 scale. Near 100, buyers won almost every day. Near 0, sellers did. Traders call above 70 "overbought" and below 30 "oversold", and the mean-reversion bet is that a rope pulled hard one way snaps back.

**The formula, one piece at a time.**

- $\Delta P_t = P_t - P_{t-1}$, today's price change.
- $G_t = \max(\Delta P_t, 0)$ and $L_t = \max(-\Delta P_t, 0)$. A day is either a gain or a loss, never both.
- $\bar G_t$ and $\bar L_t$ are averages of those, using **Wilder's smoothing**: $\bar G_t = \bar G_{t-1} + \frac{1}{n}(G_t - \bar G_{t-1})$. That is an exponential average with $\alpha = 1/n$, so recent days count more but old days never drop out completely.
- $RS = \bar G / \bar L$ and $RSI = 100 - \dfrac{100}{1 + RS}$.

In [9]:
def wilder(series, window) -> pd.Series:
    # Wilder's smoothing is an EMA with alpha = 1/n. 
    return series.ewm(alpha = 1/window, adjust=False, min_periods=window).mean()

def rsi(close, window=RSI_WINDOW) -> pd.Series:
    change = close.diff()
    gain = change.clip(lower=0)
    loss = (-change).clip(lower=0)
    rs = wilder(gain, window) / wilder(loss, window)
    return 100 - 100 / (1+rs)

### 3.2 MACD

**Intuition.** Two runners chase the price. The fast one (a 12-day average) reacts quickly and the slow one (a 26-day average) lags behind. In an uptrend the fast runner pulls ahead, and the **MACD line** is the gap between them. The **signal line** is a smoothed version of that gap. The **histogram** is the gap minus its own smoothed version, so it tells me whether the gap is widening (the trend is accelerating) or closing (it is fading). My rule trades the histogram crossing zero.

**The formula.**

- An exponential moving average with span $n$ uses $\alpha = \frac{2}{n+1}$: $EMA_t = \alpha P_t + (1 - \alpha) EMA_{t-1}$. 
- For the 12-day average $\alpha = 2/13 \approx 0.15$, so today's price gets 15% of the weight.
- $MACD_t = EMA_{12,t} - EMA_{26,t}$
- $Signal_t = EMA_9(MACD)_t$
- $Histogram_t = MACD_t - Signal_t$

In [10]:
def ema(series, span) -> pd.Series:
    return series.ewm(span=span, adjust=False).mean()

def macd(close, fast=MACD_FAST, slow=MACD_SLOW, signal=MACD_SIGNAL) -> pd.DataFrame:
    line = ema(close, fast) - ema(close, slow)
    signal_line = ema(line, signal)

    return pd.DataFrame(
        {
            "macd": line, 
            "signal": signal_line, 
            "histogram": line - signal_line
        }
    )

---
## <a id = "p4">4. From indicator to trade: the six stages</a> [back to top](#top)

I break down the flow into 6 columns. Do note that within this book, shorting is allowed.

| Stage | Column | Here |
|---|---|---|
| 1. price | `close` | adjusted close |
| 2. indicator | RSI or MACD histogram | Section 3 |
| 3. signal | a cross, which fires on a single day | `crosses_above`, `crosses_below` |
| 4. detected position | +1 / −1 / 0, held until the opposite signal | `hold_until_flip` |
| 5. executed position | detected, one day later | `.shift(1)` |
| 6. PnL | executed × return − cost | `long_short_book` |

### The two rules

- **RSI(14) 30/70, mean reversion.** 
    - Go long when the RSI climbs back above 30 (the selling is exhausted). 
    - Go short when it falls back below 70 (the buying is exhausted).

- **MACD(12,26,9), trend.** 
    - Go long when the histogram crosses above zero. 
    - Go short when it crosses below.

In [11]:
def crosses_above(series, level) -> pd.Series:
    # strictly below yesterday, strictly above today. 
    # Uses only past and present.
    return (series.shift(1) < level) & (series > level)

def crosses_below(series, level) -> pd.Series:
    return (series.shift(1) > level) & (series < level)

def hold_until_flip(enter_long, enter_short) -> pd.Series:
    # Stamp +1 / -1 on signal days, carry it forward until the opposite signal,
    # and stay flat before the first signal.
    stamped = (
        np.where(enter_long, 1.0, 
                 np.where(enter_short, -1.0, np.nan)
            )
    )
    return pd.Series(stamped, index=enter_long.index).ffill().fillna(0.0)

In [12]:
def rsi_reversion(close) -> pd.Series:
    strength = rsi(close, RSI_WINDOW)

    return hold_until_flip(crosses_above(strength, RSI_OVERSOLD),
                           crosses_below(strength, RSI_OVERBOUGHT))


def macd_momentum(close) -> pd.Series:
    histogram = macd(close, MACD_FAST, MACD_SLOW, MACD_SIGNAL)["histogram"]

    return hold_until_flip(crosses_above(histogram, 0.0),
                           crosses_below(histogram, 0.0))


RSI_RULE = f"RSI({RSI_WINDOW}) {RSI_OVERSOLD}/{RSI_OVERBOUGHT}"
MACD_RULE = f"MACD({MACD_FAST},{MACD_SLOW},{MACD_SIGNAL})"

RULES = {RSI_RULE: rsi_reversion, MACD_RULE: macd_momentum}

### Stage 5: the one line that keeps the backtest honest

**Intuition.** The RSI for Monday uses Monday's close, and I only know Monday's close once the market has shut. Trading on it during Monday would be like betting on a match after reading tomorrow's newspaper. The earliest I can act is Tuesday.

`.shift(1)` slides the position column down one row, so row *t* holds the position decided at *t − 1*, which is the only one I could really have held on day *t*. That is lookahead bias, and it flatters every rule.

### Stage 6: why simple returns, not log returns

**Intuition.** A stock falls from 100 to 50 and I am short. I sold at 100 and bought back at 50, so I made 50 on 100: **+50%**. The log version books $-\ln(50/100) = 0.693$, and $e^{0.693} - 1 = $ **+100%**. Every short leg gets overstated, and over seven years of shorts the error compounds. On this data it would have more than doubled AAPL-MACD's return and flipped TSM-RSI's expectancy from negative to positive. So stage 6 here is `executed × simple return`, compounded.

In [13]:
def long_short_book(close, position, cost_bp=COST_BP) -> pd.DataFrame:
    book = pd.DataFrame({"close": close, "detected_position": position}).dropna()

    book["executed_position"] = book["detected_position"].shift(1)
    book["simple_return"]     = book["close"].pct_change()
    book["turnover"]          = book["executed_position"].diff().abs().fillna(0.0)
    book["pnl"]               = (book["executed_position"] * book["simple_return"] - book["turnover"] * cost_bp / 10_000)

    return book.dropna(subset=["pnl"])

def build_book(closes, ticker, rule_name) -> pd.DataFrame:
    close = closes[ticker]
    
    return long_short_book(close, RULES[rule_name](close))

- On the flip day, `detected_position` goes from +1 to −1: the histogram crossed below zero at that close.
- `executed_position` is still +1 on that day and only turns −1 the next day. That one-row lag is stage 5.
- `turnover` is 2 on the first short day (from +1 to −1 is two units), so that day pays $2 \times 5$ bp $= 10$ bp of cost.
- `pnl` is `executed × simple_return` minus that cost. Nothing in the row uses information from a later row.

---
## <a id = "p5">5. Scoring a book</a> [back to top](#top)

### Win rate vs expectancy

**Intuition.** Imagine someone sells cheap insurance. On nine days out of ten nothing happens and they keep $1. On the tenth day there is a claim and they pay $20. Their win rate is 90%, and they lose $11 every ten days. Win rate counts *how often* you win but says nothing about *how much*.

**Expected return** weighs the wins and losses. If $p$ is the win rate, $\bar W$ the average winning trade and $\bar L$ the average losing trade:

$$E = p \cdot \bar W - (1 - p) \cdot \bar L$$

For the insurance seller, $E = 0.9 \times 1 - 0.1 \times 20 = -1.1$ per day. +ve win rate but -ve expected return at the same time.

- In practice $E$ is just the mean return per trade.

### Sharpe ratio and drawdown

- **Sharpe** is return per unit of bumpiness: the average daily return above the risk-free rate, divided by its standard deviation, scaled by $\sqrt{252}$ to a year. It is the ranking metric because it rewards making money *and* penalises the risk taken to make it.
- **Max drawdown** is the worst fall from a peak in the equity curve: the pain an investor would actually have sat through.

A **trade** is one unbroken run of the same non-zero executed position. Its return is the daily PnL compounded over that run.

In [14]:
def total_return(pnl):
    return (1 + pnl).prod() - 1.0

def sharpe_ratio(pnl, risk_free=RISK_FREE) -> float:
    excess = pnl - risk_free / TRADING_DAYS
    return excess.mean() / excess.std() * np.sqrt(TRADING_DAYS)

def max_drawdown(pnl) -> float:
    wealth = (1 + pnl).cumprod()
    return (wealth / wealth.cummax() - 1.0).min()

def trade_returns(book) -> pd.Series:
    position = book["executed_position"]

    # Identify active trades & assign a unique ID to each trade run
    in_trade = position != 0
    trade_id = (position != position.shift()).cumsum()

    daily_growth = 1 + book.loc[in_trade, "pnl"]         # daily gross growth factor for active trading days
    trade_growth = daily_growth.groupby(trade_id).prod() # compound daily growth factors per trade group
    
    return trade_growth - 1

In [15]:
def scorecard(book) -> dict:
    trades = trade_returns(book)
    return {
        "total_return"     : total_return(book["pnl"]),
        "sharpe"           : sharpe_ratio(book["pnl"]),
        "max_drawdown"     : max_drawdown(book["pnl"]),
        "win_rate"         : (trades > 0).mean(),
        "expected_return"  : trades.mean(),
        "n_trades"         : len(trades),
        "hold_return"      : total_return(book["simple_return"]),
        "hold_sharpe"      : sharpe_ratio(book["simple_return"]),
    }

---
## <a id = "p6">6. The scan</a> [back to top](#top)

Every ticker against every rule: 8 × 2 = 16 books, each scored the same way and ranked by Sharpe.

In [16]:
def scan(closes, rules) -> pd.DataFrame:
    rows = [
        {
            "ticker": ticker, 
            "rule": rule_name, 
            **scorecard(build_book(closes, ticker, rule_name))
        }
        for ticker in closes.columns
        for rule_name in rules
    ]

    return pd.DataFrame(rows).sort_values("sharpe", ascending=False, ignore_index=True)

results = scan(closes, RULES)

PERCENT_COLUMNS = ["total_return", "max_drawdown", "win_rate", "expected_return", "hold_return"]

display(results.round(3))

,ticker,rule,total_return,sharpe,max_drawdown,win_rate,expected_return,n_trades,hold_return,hold_sharpe
0,AAPL,"MACD(12,26,9)",2.620,0.717,-0.284,0.461,0.013,115,6.054,1.048
1,TSLA,"MACD(12,26,9)",1.948,0.521,-0.739,0.321,0.018,134,21.885,1.058
2,TSM,"MACD(12,26,9)",0.307,0.209,-0.386,0.369,0.004,130,5.685,1.014
3,JPM,"MACD(12,26,9)",0.010,0.075,-0.400,0.341,0.001,132,2.092,0.645
4,JPM,RSI(14) 30/70,-0.178,-0.030,-0.641,0.647,0.000,17,2.092,0.645
5,AAPL,RSI(14) 30/70,-0.256,-0.072,-0.549,0.611,0.003,18,6.054,1.048
6,GOOGL,RSI(14) 30/70,-0.247,-0.100,-0.564,0.667,0.003,12,4.576,0.992
7,TSLA,RSI(14) 30/70,-0.816,-0.203,-0.919,0.571,0.014,14,21.885,1.058
8,GOOGL,"MACD(12,26,9)",-0.387,-0.210,-0.517,0.336,-0.002,146,4.576,0.992
9,AMZN,"MACD(12,26,9)",-0.578,-0.328,-0.767,0.400,-0.005,130,10.291,1.243


### The headline: what each ranking would crown

The 2022 version ranked combinations by win rate. Here are the top three under each ranking.

In [17]:
HEADLINE_COLUMNS = ["ticker", "rule", "win_rate", "expected_return", "total_return", "sharpe"]

by_win_rate = results.sort_values("win_rate", ascending=False).head(3)
by_sharpe = results.head(3)

pd.concat({"ranked by win rate": by_win_rate[HEADLINE_COLUMNS].reset_index(drop=True),
           "ranked by Sharpe": by_sharpe[HEADLINE_COLUMNS].reset_index(drop=True)}).round(3)

ticker           rule  win_rate  expected_return  total_return  sharpe
ranked by win rate 0    TSM  RSI(14) 30/70     0.882           -0.005        -0.568  -0.349
                   1   MSFT  RSI(14) 30/70     0.750           -0.062        -0.747  -0.688
                   2  GOOGL  RSI(14) 30/70     0.667            0.003        -0.247  -0.100
ranked by Sharpe   0   AAPL  MACD(12,26,9)     0.461            0.013         2.620   0.717
                   1   TSLA  MACD(12,26,9)     0.321            0.018         1.948   0.521
                   2    TSM  MACD(12,26,9)     0.369            0.004         0.307   0.209

**Reading the two tables.**

- Ranked by win rate: top 3 are all RSI books that **lost money**. 
    - TSM on RSI won 88% of its trades and still lost about 57% of its capital, with-ve expected value: many small wins and a few large losses.

- Ranked by Sharpe: AAPL on MACD wins. 
    - It won **under half** its trades and returned about 2.62x, because its winners were much bigger than its losers.

- So the two rankings don't just disagree, they point in opposite directions. 
- That is the case for ranking by Sharpe and reporting expectancy beside it.

In [30]:
# Win rate on x, Sharpe on y, one point per book. If win rate were a good guide,
# the points would rise from left to right.
win_rate_vs_sharpe = results.copy()
win_rate_vs_sharpe["win_rate_pct"] = 100 * win_rate_vs_sharpe["win_rate"]

correlation = results["win_rate"].corr(results["sharpe"])
print(f"correlation of win rate with Sharpe: {correlation:+.2f}")

wr_vs_sharpe_plot = (
    ggplot(win_rate_vs_sharpe, aes(x="win_rate_pct", y="sharpe", color="rule"))
    + geom_hline(yintercept=0, linetype='dashed', color='black')
    + geom_point()
    + geom_text(aes(label="ticker"))
    + scale_color_manual(values={RSI_RULE: RED, MACD_RULE: TEAL})
    + labs(title="A higher win rate came with a worse Sharpe ratio", x="win rate (%)", y="Sharpe ratio",)
    + house_theme
    + ggsize(1000, 500)
)

wr_vs_sharpe_plot.show()

correlation of win rate with Sharpe: -0.24


AAPL 
 
 
 
 
 
 
 TSLA 
 
 
 
 
 
 
 TSM 
 
 
 
 
 
 
 JPM 
 
 
 
 
 
 
 JPM 
 
 
 
 
 
 
 AAPL 
 
 
 
 
 
 
 GOOGL 
 
 
 
 
 
 
 TSLA 
 
 
 
 
 
 
 GOOGL 
 
 
 
 
 
 
 AMZN 
 
 
 
 
 
 
 TSM 
 
 
 
 
 
 
 NVDA 
 
 
 
 
 
 
 AMZN 
 
 
 
 
 
 
 MSFT 
 
 
 
 
 
 
 MSFT 
 
 
 
 
 
 
 NVDA 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 35 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 45 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 55 
 
 
 
 
 
 
 60 
 
 
 
 
 
 
 65 
 
 
 
 
 
 
 70 
 
 
 
 
 
 
 75 
 
 
 
 
 
 
 80 
 
 
 
 
 
 
 85 
 
 
 
 
 
 
 90 
 
 
 
 
 
 
 
 
 -1.4 
 
 
 
 
 
 
 -1.2 
 
 
 
 
 
 
 -1 
 
 
 
 
 
 
 -0.8 
 
 
 
 
 
 
 -0.6 
 
 
 
 
 
 
 -0.4 
 
 
 
 
 
 
 -0.2 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 0.2 
 
 
 
 
 
 
 0.4 
 
 
 
 
 
 
 0.6 
 
 
 
 
 
 
 0.8 
 
 
 
 
 
 
 
 
 A higher win rate came with a worse Sharpe ratio 
 
 
 
 
 Sharpe ratio 
 
 
 
 
 win rate (%) 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 a 
 
 
 
 
 
 
 
 RSI(14) 30/70 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 a 
 
 
 
 
 
 
 
 MACD(12,26,9)

---
## <a id = "p7">7. Inside two books</a> [back to top](#top)

The numbers say what happened. The charts show why. Green shading is a long stretch, red a short one, and the triangles mark each entry.

In [31]:
SIDE_COLORS = {"long": GREEN, "short": RED}


def position_spans(book) -> pd.DataFrame:
    # One row per trade: where it starts, where it ends, which side it is on.
    position = book["executed_position"]

    # Identify active trades & assign a unique ID to each trade run
    in_trade = position != 0
    trade_id = (position != position.shift()).cumsum()

    trades = book.loc[in_trade].copy()
    trades["date"]     = trades.index
    trades["trade_id"] = trade_id[in_trade]

    spans = trades.groupby("trade_id").agg(
        start=("date", "first"),
        end=("date", "last"),
        side=("executed_position", "first"),
        entry_price=("close", "first"),
    )
    spans["side"]   = np.where(spans["side"] > 0, "long", "short")
    spans["bottom"] = book["close"].min()   # shade each trade across the full
    spans["top"]    = book["close"].max()   # height of the price line

    return spans


def price_panel(book, title, subtitle):
    spans = position_spans(book)
    prices = pd.DataFrame({"date": book.index, "close": book["close"].values})

    return (
        ggplot()
        + geom_rect(aes(xmin="start", xmax="end", ymin="bottom", ymax="top", fill="side"),
                    data=spans, alpha=0.15, size=0)
        + geom_line(aes(x="date", y="close"), data=prices, color=NAVY, size=0.6)
        + geom_point(aes(x="start", y="entry_price", color="side", shape="side"),
                     data=spans, size=3)
        + scale_fill_manual(values=SIDE_COLORS)
        + scale_color_manual(values=SIDE_COLORS)
        + scale_shape_manual(values={"long": 17, "short": 25})
        + labs(title=title, subtitle=subtitle, x="", y="adjusted close ($)")
        + house_theme
    )

In [32]:
def macd_panel(close):
    lines = macd(close)
    lines["date"] = lines.index
    lines_LONG = lines.melt(id_vars="date", value_vars=["macd", "signal"], var_name="series", value_name="value")

    return (
        ggplot()
        + geom_bar(aes(x="date", y="histogram"), data=lines, stat="identity", fill="#B8C2CC")
        + geom_line(aes(x="date", y="value", color="series"), data=lines_LONG, size=0.5)
        + geom_hline(yintercept=0, color=GREY)
        + scale_color_manual(values={"macd": NAVY, "signal": RED})
        + labs(x="", y="MACD ($)")
        + house_theme
    )


def rsi_panel(close):
    strength = rsi(close).dropna()
    strength = pd.DataFrame({"date": strength.index, "rsi": strength.values})

    return (
        ggplot(strength, aes(x="date", y="rsi"))
        + geom_line(color=NAVY, size=0.5)
        + geom_hline(yintercept=RSI_OVERSOLD, linetype="dashed", color=GREEN)
        + geom_hline(yintercept=RSI_OVERBOUGHT, linetype="dashed", color=RED)
        + labs(x="", y="RSI")
        + house_theme
    )

In [ ]:
def book_figure(closes, ticker, rule_name, indicator_panel):
    book = build_book(closes, ticker, rule_name)
    card = scorecard(book)

    title    = f"{ticker} on {rule_name}"
    subtitle = (
        f"return {card['total_return']:+.0%} · Sharpe {card['sharpe']:.2f} · "
        f"win rate {card['win_rate']:.0%} · {card['n_trades']} trades"
    )

    top_panel    = price_panel(book, title, subtitle)
    bottom_panel = indicator_panel(closes[ticker])

    return gggrid([top_panel, bottom_panel], ncol=1, heights=[2, 1], sharex="all") + ggsize(1000, 600)


best_book_figure = book_figure(closes, "AAPL", MACD_RULE, macd_panel)

# This image is the one shown at the top of the README.
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
ggsave(best_book_figure, "scan_AAPL_MACD.png", path=str(FIGURE_DIR))

best_book_figure

Fontconfig error: Cannot load default config file: No such file: (null)


**AAPL on MACD, the top Sharpe.** 
- High activity, low win rate
    - Triggered 115 trades over seven years with a 46% win rate, losing on more than half of its trades due to frequent whipsaws during choppy sideways moves.   
    
- Strict downside control
    - Losses were kept small, with the single worst trade capped at −10%.
    
- Riding major trends
    - Huge wins came from riding real market momentum—such as the post-COVID rebound (+29%), the late-2018 selloff on the short side (+17%), and the late-2019 rally (+16%).
    
- The takeaway
    - Classic trend-following behavior. 
    - A low win rate is completely offset by asymmetry—a few huge trend trades easily pay for many small whipsaws, generating a +262% total return and a top 0.72 Sharpe ratio. 

In [34]:
# ggsave(book_figure(closes, "TSM", RSI_RULE, rsi_panel), "scan_TSM_RSI.png", path=str(FIGURE_DIR))
book_figure(closes, "TSM", RSI_RULE, rsi_panel)

**TSM on RSI, the highest win rate.** 
- High win rate, small gains 
    - Achieved an 88% win rate across 17 trades, but 15 of those wins were modest (2% to 15%).

- The structural flaw
    - Short signals triggered whenever RSI crossed below 70, locking the portfolio short until RSI hit oversold levels (30). 
    - In a stock surging nearly 7x, reaching RSI 30 took years.   
    
- Catastrophic short losses
    - Two short positions did all the damage: losing 42% from 2016 to 2018, and 73% from 2019 to 2021 while trapped straight through TSM's 2020 rally.   
    
- The takeaway
    - A classic high-win-rate trap. 
    - Asymmetric downside risk meant 15 small gains were completely destroyed by 2 open-ended short losses, ending in a −57% overall return and a −0.35 Sharpe ratio.

---
## <a id = "p8">8. What I learned</a> [back to top](#top)

### Lesson 1: the thesis and the data can disagree, and the backtest doesn't have to agree with my story

The strategy was designed as trend-following momentum: buy strength, sell weakness. In 2022, when I grid-searched combinations, the best-ranked results often came from RSI, a mean-reverting, counter-trend indicator. A trend-following design with a mean-reverting winner was the most interesting tension in that work.

This rebuild, on a different and public universe (eight mega-caps, 2015 to 2021), did **not** reproduce that. MACD holds the top four spots and every RSI book has a negative Sharpe ratio. I take that as the same lesson from the other side: no indicator is universally right, and which one "wins" depends on the instruments and the period. 

A rule is a hypothesis about a market, not a fact.

### Lesson 2: RSI alone whipsaws in trends

RSI fires counter-trend entries during clean trends and bleeds on them. The split below shows where the RSI books made and lost money, long side against short side.

In [23]:
def pnl_by_side(book) -> pd.Series:
    # Compounded return earned while long, and while short, over the same book.
    side = np.sign(book["executed_position"]).map({1.0: "long", -1.0: "short", 0.0: "flat"})
    side_growth = (1 + book["pnl"]).groupby(side).prod()

    return side_growth.drop("flat", errors="ignore") - 1


side_returns = {}
for ticker in closes.columns:
    rsi_book = build_book(closes, ticker, RSI_RULE)
    side_returns[ticker] = pnl_by_side(rsi_book)

rsi_sides = pd.DataFrame(side_returns).T
rsi_sides = rsi_sides.sort_values("short")

rsi_sides.style.format("{:+.0%}")

executed_position,long,short
NVDA,+7%,-99%
TSLA,+285%,-95%
MSFT,+87%,-86%
AMZN,+103%,-85%
TSM,+101%,-79%
AAPL,+157%,-71%
GOOGL,+130%,-67%
JPM,+88%,-56%


**Reading the table.** On all eight tickers the RSI book's long side made money and its short side lost it, between 56% and 99% of capital. Shorting "overbought" in a bull market means shorting strength. In 2022 this led me to the idea of using MACD as a **trend filter** to veto RSI's worst trades: take the mean-reverting signal only when the trend signal agrees. I have kept that as an idea, not built it into this scanner.

### Lesson 3: win rate is not enough

A strategy can win most of its trades and still lose money. TSM-RSI won 88% and lost more than 57% its capital. AAPL-MACD won under half and returned about +260%. That is why the scanner ranks by Sharpe and reports expected returns rather than win rate.

---
## <a id = "p9">9. Limitations of strategy</a> [back to top](#top)

- **In-sample only, no train/test split.** The parameters are textbook defaults, not tuned, but nothing here is validated out of sample.

- **Simple technical signals mostly lose here.** Most books have a negative Sharpe ratio, and even the best one trails buy-and-hold. That is the honest result. The value of the project is the framework and the ranking discipline, not a strategy.

- **Flat cost model.** 5 bp per unit of turnover, with no spread, slippage or market impact.

- **Survivorship bias.** The universe is eight names that did well. A real study needs a point-in-time universe that includes delistings. Access to databases like CRSP help to circumvent this issue of survivorship bias.